# Auditoria Integrada Final — Projeto ENEM (UF PA)

## Finalidade

Este notebook realiza uma auditoria **somente de leitura** das camadas Gold, Analytic e Consumo para a UF configurada. O objetivo é verificar a integridade dos arquivos e metadados já produzidos, sem reprocessar dados nem modificar os Parquets dessas camadas.

A auditoria está organizada em quatro frentes:

1. **Gold:** integridade do manifesto, existência dos arquivos, contagens de registros e referências territoriais por ano;
2. **Analytic:** produtos esperados, existência dos arquivos, unicidade das chaves e preenchimento territorial, além da cobertura de 2024 e 2025;
3. **Consumo:** produtos esperados, existência dos arquivos, unicidade das chaves e cobertura territorial de 2024 e 2025;
4. **Metadados:** presença dos manifestos, catálogo e auditoria de Consumo.

Ao final, os testes são consolidados em uma tabela integrada e exportados para CSV. A exportação do relatório é a única gravação realizada por este notebook.

**Importante:** a aprovação indica que os critérios implementados nas células foram atendidos. Ela não substitui uma auditoria semântica completa de todas as variáveis ou regras de negócio.


## 1. Configuração

A célula seguinte carrega os módulos e bibliotecas necessários e define a UF auditada (`UF = "PA"`).

Os caminhos dos arquivos de controle são construídos a partir de `cfg.METADATA_DIR`, mantendo a organização de metadados definida no projeto:

- manifesto Gold;
- manifesto Analytic;
- manifesto Consumo;
- catálogo de Consumo;
- auditoria anterior de Consumo.

Se a UF ou os nomes dos arquivos de metadados forem diferentes no ambiente de execução, ajuste os parâmetros nesta seção. Os caminhos dos Parquets auditados são obtidos dos próprios manifestos, não são reconstruídos manualmente neste notebook.


In [1]:
from _setup import *

import pandas as pd
import duckdb
from IPython.display import display
from enem_pipeline import config as cfg

UF = "PA"
GOLD_MANIFEST = Path(cfg.METADATA_DIR) / f"manifesto_gold_{UF}.csv"
ANALYTIC_MANIFEST = Path(cfg.METADATA_DIR) / f"manifesto_analytic_{UF}.csv"
CONSUMO_MANIFEST = Path(cfg.METADATA_DIR) / f"manifesto_consumo_{UF}.csv"
CONSUMO_CATALOG = Path(cfg.METADATA_DIR) / f"catalogo_consumo_{UF}.csv"
CONSUMO_AUDIT = Path(cfg.METADATA_DIR) / f"auditoria_consumo_{UF}.csv"

print("UF:", UF)
print("Metadados:", cfg.METADATA_DIR)

UF: PA
Metadados: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata


## 2. Funções auxiliares

As funções desta seção padronizam a leitura e a apresentação dos testes:

- `ler_manifesto`: verifica se o manifesto existe e o carrega como DataFrame, usando `;` como separador;
- `contar_parquet`: conta os registros de um arquivo Parquet por meio do DuckDB;
- `teste`: registra o nome do teste, o resultado booleano (`ok`), o valor obtido e o esperado;
- `mostrar_auditoria`: exibe a tabela de resultados, contabiliza os testes aprovados e destaca as linhas reprovadas.

Essas funções não alteram os arquivos de dados. Quando um arquivo obrigatório não existe, as funções de leitura interrompem a execução com erro explícito, em vez de tratar a ausência como uma aprovação.


In [2]:
def ler_manifesto(path):
    if not Path(path).is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    return pd.read_csv(path, sep=";")


def contar_parquet(path):
    path = Path(str(path))
    if not path.is_file():
        raise FileNotFoundError(path)
    with duckdb.connect() as con:
        return int(con.execute(
            "SELECT COUNT(*) FROM read_parquet(?)", [str(path)]
        ).fetchone()[0])


def teste(nome, ok, obtido=None, esperado=None):
    return {
        "teste": nome, "ok": bool(ok),
        "obtido": obtido, "esperado": esperado
    }


def mostrar_auditoria(df, titulo):
    print(titulo)
    display(df)
    falhas = df.loc[~df["ok"]]
    print(f"{int(df['ok'].sum())}/{len(df)} testes aprovados")
    if not falhas.empty:
        print("PENDÊNCIAS:")
        display(falhas)

## 3. Leitura dos manifestos

Esta etapa carrega os três manifestos que descrevem os produtos das camadas Gold, Analytic e Consumo.

As dimensões exibidas (`shape`) permitem uma primeira conferência da quantidade de registros e colunas de cada manifesto. As amostras (`head`) servem para inspecionar a estrutura e os campos de referência antes da auditoria detalhada.

A leitura bem-sucedida confirma apenas que os arquivos de manifesto foram localizados e interpretados. A validação de conteúdo, arquivos associados e chaves ocorre nas seções seguintes.


In [3]:
gold = ler_manifesto(GOLD_MANIFEST)
analytic = ler_manifesto(ANALYTIC_MANIFEST)
consumo = ler_manifesto(CONSUMO_MANIFEST)

print("Gold:", gold.shape)
print("Analytic:", analytic.shape)
print("Consumo:", consumo.shape)

display(gold.head(5))
display(analytic.head(9))
display(consumo.head(4))

Gold: (48, 10)
Analytic: (9, 9)
Consumo: (4, 9)


,camada,uf,ano,base,tipo_referencia_territorial,unidade_integracao,join_individual,arquivo,arquivo_gold,n_registros
0,gold,PA,1998,enem,RESIDENCIA,ano_municipio_tipo_referencia,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,288
1,gold,PA,1999,enem,RESIDENCIA,ano_municipio_tipo_referencia,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,6518
2,gold,PA,2000,enem,RESIDENCIA,ano_municipio_tipo_referencia,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,6219
3,gold,PA,2001,enem,ESC,ano_municipio_tipo_referencia,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,24295
4,gold,PA,2002,enem,ESC,ano_municipio_tipo_referencia,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,32233


,camada,uf,produto,granularidade,periodo,arquivo_analytic,n_linhas,n_colunas,tamanho_mb
0,analytic,PA,demografia_municipal,ano_municipio,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,6464,11,0.123483
1,analytic,PA,faixa_etaria_municipal,ano_municipio_faixa_etaria,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,63836,7,0.142610
2,analytic,PA,presenca_municipal,ano_municipio,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,6464,11,0.060381
3,analytic,PA,desempenho_redacao_municipal,ano_municipio,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,6464,13,0.197515
4,analytic,PA,desempenho_objetiva_municipal,ano_municipio,1998-2008,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,1454,13,0.057270
5,analytic,PA,desempenho_areas_municipal,ano_municipio,2009-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,5010,41,0.693615
6,analytic,PA,escola_municipal,ano_municipio,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,6464,17,0.155934
7,analytic,PA,desempenho_grupos_municipal,ano_municipio_grupo_disciplina,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,337214,16,8.590934
8,analytic,PA,desempenho_sexo_faixa_etaria_municipal,ano_municipio_sexo_faixa_etaria_disciplina,2001-2023,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,451205,17,10.113083


,camada,uf,produto,granularidade,periodo,arquivo_consumo,n_linhas,n_colunas,tamanho_mb
0,consumo,PA,painel_municipal,ano_municipio_referencia,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/c...,6464,82,1.185326
1,consumo,PA,faixa_etaria_municipal,ano_municipio_referencia_faixa_etaria,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/c...,63836,7,0.144585
2,consumo,PA,desempenho_grupos_municipal,ano_municipio_referencia_grupo_disciplina,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/c...,337214,16,8.573952
3,consumo,PA,desempenho_sexo_faixa_etaria_municipal,ano_municipio_referencia_sexo_faixa_etaria_dis...,2001-2023,/home/akel/PycharmProjects/Projeto_ENEM/data/c...,451205,17,10.102821


## 4. Auditoria Gold

A auditoria Gold verifica os seguintes critérios:

1. Presença das colunas obrigatórias no manifesto: `ano`, `base`, `tipo_referencia_territorial`, `arquivo_gold` e `n_registros`;
2. Ausência de duplicidades na combinação `ano`, `base` e `tipo_referencia_territorial`;
3. Quantidade de fontes registrada no manifesto: **48**, conforme o critério fixado no código;
4. Existência de cada arquivo Parquet indicado no manifesto;
5. Igualdade entre `n_registros` declarado no manifesto e a contagem real do Parquet;
6. Presença das referências territoriais esperadas em cada ano.

A regra territorial codificada é: `RESIDENCIA` para 1998–2000; `ESC` para 2001–2007; e `ESC` e `PROVA` para 2008–2025. A verificação de cobertura usa inclusão de referências esperadas (`issubset`), enquanto a quantidade total de fontes é verificada separadamente.

As tabelas intermediárias permitem localizar o ano, a base ou o arquivo responsável por uma eventual falha. A aprovação da seção depende dos testes reunidos em `df_gold_audit`.


### 4.1 Critérios de contagem e cobertura territorial

A auditoria aplica dois controles distintos: a contagem total de fontes do manifesto (`48`) e a presença das referências territoriais esperadas em cada ano. O primeiro é uma quantidade fixa; o segundo verifica se cada conjunto esperado está contido nas referências encontradas.

Essa distinção é importante porque a cobertura territorial não é usada para calcular nem substituir a quantidade total de fontes.


In [4]:
gold_tests = []
required = {"ano", "base", "tipo_referencia_territorial", "arquivo_gold", "n_registros"}
missing = required - set(gold.columns)
gold_tests.append(teste("gold_colunas_obrigatorias", not missing, sorted(missing), []))

if not missing:
    gold["ano"] = pd.to_numeric(gold["ano"], errors="raise").astype(int)
    gold["n_registros"] = pd.to_numeric(gold["n_registros"], errors="raise").astype(int)

    dup = int(gold.duplicated(["ano", "base", "tipo_referencia_territorial"]).sum())
    gold_tests.append(teste("gold_manifesto_sem_duplicidade", dup == 0, dup, 0))
    gold_tests.append(teste("gold_48_fontes", len(gold) == 48, len(gold), 48))

    rows = []
    for _, r in gold.iterrows():
        p = Path(r["arquivo_gold"])
        existe = p.is_file()
        n = contar_parquet(p) if existe else None
        rows.append({
            "ano": r["ano"], "base": r["base"],
            "referencia": r["tipo_referencia_territorial"],
            "arquivo_existe": existe, "n_manifesto": r["n_registros"],
            "n_parquet": n, "contagem_ok": existe and n == r["n_registros"]
        })
    df_gold_arquivos = pd.DataFrame(rows)
    display(df_gold_arquivos)
    gold_tests.append(teste(
        "gold_arquivos_e_contagens",
        bool(df_gold_arquivos["contagem_ok"].all()),
        int(df_gold_arquivos["contagem_ok"].sum()), len(df_gold_arquivos)
    ))

    esperadas = {a: {"RESIDENCIA"} for a in [1998, 1999, 2000]}
    esperadas.update({a: {"ESC"} for a in range(2001, 2008)})
    esperadas.update({a: {"ESC", "PROVA"} for a in range(2008, 2026)})
    cobertura = []
    for ano, refs in esperadas.items():
        encontradas = set(gold.loc[gold["ano"].eq(ano), "tipo_referencia_territorial"].dropna())
        cobertura.append({
            "ano": ano, "esperadas": sorted(refs),
            "encontradas": sorted(encontradas), "ok": refs.issubset(encontradas)
        })
    df_gold_cobertura = pd.DataFrame(cobertura)
    display(df_gold_cobertura)
    gold_tests.append(teste(
        "gold_referencias_por_ano",
        bool(df_gold_cobertura["ok"].all()),
        int(df_gold_cobertura["ok"].sum()), len(df_gold_cobertura)
    ))

df_gold_audit = pd.DataFrame(gold_tests)
mostrar_auditoria(df_gold_audit, "Auditoria Gold")

,ano,base,referencia,arquivo_existe,n_manifesto,n_parquet,contagem_ok
0,1998,enem,RESIDENCIA,True,288,288,True
1,1999,enem,RESIDENCIA,True,6518,6518,True
2,2000,enem,RESIDENCIA,True,6219,6219,True
3,2001,enem,ESC,True,24295,24295,True
4,2002,enem,ESC,True,32233,32233,True
5,2003,enem,ESC,True,41258,41258,True
6,2004,enem,ESC,True,34715,34715,True
7,2005,enem,ESC,True,49404,49404,True
8,2006,enem,ESC,True,36223,36223,True
9,2007,enem,ESC,True,45212,45212,True


,ano,esperadas,encontradas,ok
0,1998,[RESIDENCIA],[RESIDENCIA],True
1,1999,[RESIDENCIA],[RESIDENCIA],True
2,2000,[RESIDENCIA],[RESIDENCIA],True
3,2001,[ESC],[ESC],True
4,2002,[ESC],[ESC],True
5,2003,[ESC],[ESC],True
6,2004,[ESC],[ESC],True
7,2005,[ESC],[ESC],True
8,2006,[ESC],[ESC],True
9,2007,[ESC],[ESC],True


Auditoria Gold


,teste,ok,obtido,esperado
0,gold_colunas_obrigatorias,True,[],[]
1,gold_manifesto_sem_duplicidade,True,0,0
2,gold_48_fontes,True,48,48
3,gold_arquivos_e_contagens,True,48,48
4,gold_referencias_por_ano,True,28,28


5/5 testes aprovados


## 5. Auditoria Analytic

Esta seção verifica o conjunto de produtos Analytic declarado em `analytic_keys` e confronta os nomes com o campo `produto` do manifesto.

Para cada produto, o código:

- verifica a existência do arquivo indicado em `arquivo_analytic`;
- carrega o Parquet quando o arquivo existe;
- conta linhas;
- verifica duplicidades segundo a chave específica do produto;
- conta valores nulos em `TIPO_REFERENCIA_TERRITORIAL`;
- contabiliza códigos municipais ausentes.

Em seguida, a cobertura territorial de 2024 e 2025 é comparada com as referências esperadas para cada produto. O produto `desempenho_sexo_faixa_etaria_municipal` recebe listas vazias como esperado e encontrado nesses anos, conforme a regra explicitamente implementada no código.

**Atenção à descrição original:** o texto anterior desta seção mencionava oito produtos, mas o dicionário `analytic_keys` contém **nove nomes de produtos**. O teste compara o manifesto com esse conjunto de nove nomes. Esta documentação registra o comportamento efetivo, sem alterar a regra de auditoria.


In [5]:
analytic_keys = {
    "demografia_municipal": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL"],
    "faixa_etaria_municipal": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "TP_FAIXA_ETARIA"],
    "presenca_municipal": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL"],
    "desempenho_redacao_municipal": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL"],
    "desempenho_objetiva_municipal": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL"],
    "desempenho_areas_municipal": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL"],
    "escola_municipal": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL"],
    "desempenho_grupos_municipal": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
                                    "tipo_grupo", "grupo", "disciplina"],
    "desempenho_sexo_faixa_etaria_municipal":["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL","sexo","faixa_etaria",
                                              "disciplina"]}
 
analytic_expected = set(analytic_keys)
analytic_tests = []
analytic_tests.append(teste(
    "analytic_oito_produtos",
    set(analytic["produto"]) == analytic_expected,
    sorted(set(analytic["produto"])), sorted(analytic_expected)
))

analytic_frames = {}
rows = []
for _, r in analytic.iterrows():
    nome = r["produto"]
    p = Path(r["arquivo_analytic"])
    existe = p.is_file()
    df = pd.read_parquet(p) if existe else None
    if df is not None:
        analytic_frames[nome] = df
    keys = analytic_keys.get(nome, [])
    dup = int(df.duplicated(keys).sum()) if df is not None and set(keys).issubset(df.columns) else None
    n_tipo = int(df["TIPO_REFERENCIA_TERRITORIAL"].isna().sum()) if df is not None else None
    rows.append({
        "produto": nome, "arquivo_existe": existe,
        "linhas": len(df) if df is not None else None,
        "duplicidades_chave": dup, "tipos_ausentes": n_tipo,
        "municipios_ausentes": int(df["CO_MUNICIPIO_TERRITORIO"].isna().sum()) if df is not None else None
    })

df_analytic_arquivos = pd.DataFrame(rows)
display(df_analytic_arquivos)
analytic_tests.extend([
    teste("analytic_arquivos_existentes", bool(df_analytic_arquivos["arquivo_existe"].all()),
          int(df_analytic_arquivos["arquivo_existe"].sum()), len(df_analytic_arquivos)),
    teste("analytic_chaves_unicas", bool(df_analytic_arquivos["duplicidades_chave"].fillna(-1).eq(0).all()),
          df_analytic_arquivos["duplicidades_chave"].tolist(), "todos zero"),
    teste("analytic_tipo_territorial_preenchido", bool(df_analytic_arquivos["tipos_ausentes"].fillna(-1).eq(0).all()),
          df_analytic_arquivos["tipos_ausentes"].tolist(), "todos zero")
])

refs_analytic = {
    "demografia_municipal": {"ESC", "PROVA"},
    "faixa_etaria_municipal": {"PROVA"},
    "presenca_municipal": {"ESC", "PROVA"},
    "desempenho_redacao_municipal": {"ESC", "PROVA"},
    "desempenho_areas_municipal": {"ESC", "PROVA"},
    "escola_municipal": {"ESC", "PROVA"},
    "desempenho_grupos_municipal": {"ESC", "PROVA"},
    "desempenho_sexo_faixa_etaria_municipal":{"ESC","PROVA"}}

coverage = []

for nome, refs in refs_analytic.items():
    df = analytic_frames.get(nome)
    for ano in [2024, 2025]:
        if nome=='desempenho_sexo_faixa_etaria_municipal':
            esperado, encontrado = [], []
        else:
            esperado=sorted(refs)
            encontrado=sorted(set(df.loc[df["ano"].eq(ano), "TIPO_REFERENCIA_TERRITORIAL"].dropna()) if df is not None else set())
            
        coverage.append({
            "produto": nome, "ano": ano, "esperado": esperado,
            "encontrado": encontrado, "ok": esperado == encontrado})
            
df_analytic_coverage = pd.DataFrame(coverage)
display(df_analytic_coverage)
analytic_tests.append(teste(
    "analytic_cobertura_territorial_2024_2025",
    bool(df_analytic_coverage["ok"].all()),
    int(df_analytic_coverage["ok"].sum()), len(df_analytic_coverage)
))

df_analytic_audit = pd.DataFrame(analytic_tests)
mostrar_auditoria(df_analytic_audit, "Auditoria Analytic")

,produto,arquivo_existe,linhas,duplicidades_chave,tipos_ausentes,municipios_ausentes
0,demografia_municipal,True,6464,0,0,18
1,faixa_etaria_municipal,True,63836,0,0,320
2,presenca_municipal,True,6464,0,0,18
3,desempenho_redacao_municipal,True,6464,0,0,18
4,desempenho_objetiva_municipal,True,1454,0,0,1
5,desempenho_areas_municipal,True,5010,0,0,17
6,escola_municipal,True,6464,0,0,18
7,desempenho_grupos_municipal,True,337214,0,0,1781
8,desempenho_sexo_faixa_etaria_municipal,True,451205,0,0,3078


,produto,ano,esperado,encontrado,ok
0,demografia_municipal,2024,"[ESC, PROVA]","[ESC, PROVA]",True
1,demografia_municipal,2025,"[ESC, PROVA]","[ESC, PROVA]",True
2,faixa_etaria_municipal,2024,[PROVA],[PROVA],True
3,faixa_etaria_municipal,2025,[PROVA],[PROVA],True
4,presenca_municipal,2024,"[ESC, PROVA]","[ESC, PROVA]",True
5,presenca_municipal,2025,"[ESC, PROVA]","[ESC, PROVA]",True
6,desempenho_redacao_municipal,2024,"[ESC, PROVA]","[ESC, PROVA]",True
7,desempenho_redacao_municipal,2025,"[ESC, PROVA]","[ESC, PROVA]",True
8,desempenho_areas_municipal,2024,"[ESC, PROVA]","[ESC, PROVA]",True
9,desempenho_areas_municipal,2025,"[ESC, PROVA]","[ESC, PROVA]",True


Auditoria Analytic


,teste,ok,obtido,esperado
0,analytic_oito_produtos,True,"[demografia_municipal, desempenho_areas_munici...","[demografia_municipal, desempenho_areas_munici..."
1,analytic_arquivos_existentes,True,9,9
2,analytic_chaves_unicas,True,"[0, 0, 0, 0, 0, 0, 0, 0, 0]",todos zero
3,analytic_tipo_territorial_preenchido,True,"[0, 0, 0, 0, 0, 0, 0, 0, 0]",todos zero
4,analytic_cobertura_territorial_2024_2025,True,16,16


5/5 testes aprovados


## 6. Auditoria Consumo

A seção verifica o conjunto de produtos declarado em `consumo_keys`. Para cada produto listado no manifesto, o código verifica a existência do Parquet, carrega os dados, conta linhas e calcula duplicidades segundo a chave específica.

Depois, compara as referências territoriais encontradas em 2024 e 2025 com as referências esperadas. Para `desempenho_sexo_faixa_etaria_municipal`, o código define listas vazias para os dois anos, de modo que a cobertura esperada e encontrada seja avaliada como vazia.

**Atenção à descrição original:** o texto anterior mencionava três produtos, mas `consumo_keys` contém **quatro produtos**. O teste `consumo_tres_produtos` compara o manifesto com o conjunto de quatro nomes definido no dicionário. O nome do teste e o comentário descritivo não foram alterados no código; a divergência fica registrada aqui para conferência.


In [6]:
consumo_keys = {
    "painel_municipal": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL"],
    "faixa_etaria_municipal": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL", "TP_FAIXA_ETARIA"],
    "desempenho_grupos_municipal": ["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL","tipo_grupo", "grupo",
                                    "disciplina"],
    'desempenho_sexo_faixa_etaria_municipal':["ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL","sexo","faixa_etaria",
                                              "disciplina"]}
consumo_tests = []
consumo_expected = set(consumo_keys)
consumo_tests.append(teste(
    "consumo_tres_produtos",
    set(consumo["produto"]) == consumo_expected,
    sorted(set(consumo["produto"])), sorted(consumo_expected)
))

consumo_frames = {}
rows = []
for _, r in consumo.iterrows():
    nome = r["produto"]
    p = Path(r["arquivo_consumo"])
    existe = p.is_file()
    df = pd.read_parquet(p) if existe else None
    if df is not None:
        consumo_frames[nome] = df
    keys = consumo_keys.get(nome, [])

    print(keys)
    dup = int(df.duplicated(keys).sum()) if df is not None and set(keys).issubset(df.columns) else None
    rows.append({
        "produto": nome, "arquivo_existe": existe,
        "linhas": len(df) if df is not None else None,
        "duplicidades_chave": dup
    })

df_consumo_arquivos = pd.DataFrame(rows)
display(df_consumo_arquivos)
consumo_tests.extend([
    teste("consumo_arquivos_existentes", bool(df_consumo_arquivos["arquivo_existe"].all()),
          int(df_consumo_arquivos["arquivo_existe"].sum()), len(df_consumo_arquivos)),
    teste("consumo_chaves_unicas", bool(df_consumo_arquivos["duplicidades_chave"].fillna(-1).eq(0).all()),
          df_consumo_arquivos["duplicidades_chave"].tolist(), "todos zero")
])

refs_consumo = {
    "painel_municipal": {"ESC", "PROVA"},
    "faixa_etaria_municipal": {"PROVA"},
    "desempenho_grupos_municipal": {"ESC", "PROVA"},
    "desempenho_sexo_faixa_etaria_municipal":{"ESC","PROVA"}}

coverage = []
for nome, refs in refs_consumo.items():
    df = consumo_frames.get(nome)
    for ano in [2024, 2025]:
        # Trata o caso onde o produto não existe para estes anos, mantendo padrão de lista
        if nome == 'desempenho_sexo_faixa_etaria_municipal':
            esperado, encontrado = [], []
        else:
            esperado = sorted(refs)
            encontrado = sorted(set(df.loc[df["ano"].eq(ano), "TIPO_REFERENCIA_TERRITORIAL"].dropna())) if df is not None and "ano" in df.columns else []
            
        coverage.append({
            "produto": nome, 
            "ano": ano, 
            "esperado": esperado,
            "encontrado": encontrado, 
            "ok": encontrado == esperado
        })
        
df_consumo_coverage = pd.DataFrame(coverage)
display(df_consumo_coverage)
consumo_tests.append(teste(
    "consumo_cobertura_territorial_2024_2025",
    bool(df_consumo_coverage["ok"].all()),
    int(df_consumo_coverage["ok"].sum()), len(df_consumo_coverage)
))

df_consumo_audit = pd.DataFrame(consumo_tests)
mostrar_auditoria(df_consumo_audit, "Auditoria Consumo")

['ano', 'CO_MUNICIPIO_TERRITORIO', 'TIPO_REFERENCIA_TERRITORIAL']
['ano', 'CO_MUNICIPIO_TERRITORIO', 'TIPO_REFERENCIA_TERRITORIAL', 'TP_FAIXA_ETARIA']
['ano', 'CO_MUNICIPIO_TERRITORIO', 'TIPO_REFERENCIA_TERRITORIAL', 'tipo_grupo', 'grupo', 'disciplina']
['ano', 'CO_MUNICIPIO_TERRITORIO', 'TIPO_REFERENCIA_TERRITORIAL', 'sexo', 'faixa_etaria', 'disciplina']


,produto,arquivo_existe,linhas,duplicidades_chave
0,painel_municipal,True,6464,0
1,faixa_etaria_municipal,True,63836,0
2,desempenho_grupos_municipal,True,337214,0
3,desempenho_sexo_faixa_etaria_municipal,True,451205,0


,produto,ano,esperado,encontrado,ok
0,painel_municipal,2024,"[ESC, PROVA]","[ESC, PROVA]",True
1,painel_municipal,2025,"[ESC, PROVA]","[ESC, PROVA]",True
2,faixa_etaria_municipal,2024,[PROVA],[PROVA],True
3,faixa_etaria_municipal,2025,[PROVA],[PROVA],True
4,desempenho_grupos_municipal,2024,"[ESC, PROVA]","[ESC, PROVA]",True
5,desempenho_grupos_municipal,2025,"[ESC, PROVA]","[ESC, PROVA]",True
6,desempenho_sexo_faixa_etaria_municipal,2024,[],[],True
7,desempenho_sexo_faixa_etaria_municipal,2025,[],[],True


Auditoria Consumo


,teste,ok,obtido,esperado
0,consumo_tres_produtos,True,"[desempenho_grupos_municipal, desempenho_sexo_...","[desempenho_grupos_municipal, desempenho_sexo_..."
1,consumo_arquivos_existentes,True,4,4
2,consumo_chaves_unicas,True,"[0, 0, 0, 0]",todos zero
3,consumo_cobertura_territorial_2024_2025,True,8,8


4/4 testes aprovados


## 7. Metadados e auditoria anterior

Esta etapa verifica a existência dos cinco arquivos de controle utilizados ou referenciados pelo fluxo:

- manifesto Gold;
- manifesto Analytic;
- manifesto Consumo;
- catálogo de Consumo;
- auditoria anterior de Consumo.

O teste registra o caminho de cada arquivo e o resultado da verificação de existência. Ele não reabre nem revalida o conteúdo desses cinco arquivos nesta seção; a validação dos manifestos principais já ocorre nas seções anteriores.


In [7]:
metadata_tests = []
for nome, path in {
    "manifesto_gold": GOLD_MANIFEST,
    "manifesto_analytic": ANALYTIC_MANIFEST,
    "manifesto_consumo": CONSUMO_MANIFEST,
    "catalogo_consumo": CONSUMO_CATALOG,
    "auditoria_consumo": CONSUMO_AUDIT,
}.items():
    metadata_tests.append(teste(f"{nome}_existe", path.is_file(), str(path), "arquivo existente"))

df_metadata_audit = pd.DataFrame(metadata_tests)
mostrar_auditoria(df_metadata_audit, "Auditoria dos metadados")

Auditoria dos metadados


,teste,ok,obtido,esperado
0,manifesto_gold_existe,True,/home/akel/PycharmProjects/Projeto_ENEM/data/m...,arquivo existente
1,manifesto_analytic_existe,True,/home/akel/PycharmProjects/Projeto_ENEM/data/m...,arquivo existente
2,manifesto_consumo_existe,True,/home/akel/PycharmProjects/Projeto_ENEM/data/m...,arquivo existente
3,catalogo_consumo_existe,True,/home/akel/PycharmProjects/Projeto_ENEM/data/m...,arquivo existente
4,auditoria_consumo_existe,True,/home/akel/PycharmProjects/Projeto_ENEM/data/m...,arquivo existente


5/5 testes aprovados


## 8. Resultado integrado

Os DataFrames de testes das quatro frentes — Gold, Analytic, Consumo e metadados — são concatenados em `auditoria_integrada`.

A tabela consolidada apresenta, para cada teste:
- `teste`: identificador do critério;
- `ok`: resultado booleano;
- `obtido`: valor observado;
- `esperado`: condição ou valor esperado.

O resumo contabiliza o total de testes aprovados e reprovados. A mensagem final distingue dois estados:

- **AUDITORIA INTEGRADA APROVADA:** todos os testes registrados apresentam `ok = True`;
- **AUDITORIA INTEGRADA COM PENDÊNCIAS:** pelo menos um teste falhou e deve ser investigado.

A aprovação é restrita aos critérios implementados neste notebook e à UF definida na configuração.


In [8]:
auditoria_integrada = pd.concat(
    [df_gold_audit, df_analytic_audit, df_consumo_audit, df_metadata_audit],
    ignore_index=True
)
display(auditoria_integrada)

n_ok = int(auditoria_integrada["ok"].sum())
n_total = len(auditoria_integrada)
print(f"Testes aprovados: {n_ok}/{n_total}")
print(f"Testes reprovados: {n_total - n_ok}/{n_total}")

if n_ok == n_total:
    print("AUDITORIA INTEGRADA APROVADA — UF PA")
else:
    print("AUDITORIA INTEGRADA COM PENDÊNCIAS — revisar os testes False")

,teste,ok,obtido,esperado
0,gold_colunas_obrigatorias,True,[],[]
1,gold_manifesto_sem_duplicidade,True,0,0
2,gold_48_fontes,True,48,48
3,gold_arquivos_e_contagens,True,48,48
4,gold_referencias_por_ano,True,28,28
5,analytic_oito_produtos,True,"[demografia_municipal, desempenho_areas_munici...","[demografia_municipal, desempenho_areas_munici..."
6,analytic_arquivos_existentes,True,9,9
7,analytic_chaves_unicas,True,"[0, 0, 0, 0, 0, 0, 0, 0, 0]",todos zero
8,analytic_tipo_territorial_preenchido,True,"[0, 0, 0, 0, 0, 0, 0, 0, 0]",todos zero
9,analytic_cobertura_territorial_2024_2025,True,16,16


Testes aprovados: 19/19
Testes reprovados: 0/19
AUDITORIA INTEGRADA APROVADA — UF PA


## 9. Exportar relatório final

A tabela `auditoria_integrada` é exportada para `auditoria_integrada_PA.csv` (ou para a UF definida em `UF`), dentro de `cfg.METADATA_DIR`.

O arquivo utiliza:
- separador `;`;
- ausência de índice;
- codificação `utf-8-sig`, adequada para abertura em ferramentas que reconhecem BOM, como algumas versões do Excel.

Esta é a saída persistente produzida pelo notebook. A exportação registra os resultados da execução atual; ela não altera os manifestos nem os Parquets auditados.


In [9]:
RELATORIO_FINAL = Path(cfg.METADATA_DIR) / f"auditoria_integrada_{UF}.csv"
auditoria_integrada.to_csv(
    RELATORIO_FINAL, sep=";", index=False, encoding="utf-8-sig"
)
print("Relatório salvo em:", RELATORIO_FINAL)

Relatório salvo em: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata/auditoria_integrada_PA.csv


## 10. Critérios de encerramento

Segundo os critérios documentados nesta seção, a auditoria deve ser considerada aprovada quando:

1. Os 48 registros esperados do manifesto Gold são verificados, os arquivos existem e as contagens conferem;
2. Os produtos Analytic declarados no código existem, têm chaves únicas e atendem aos testes de preenchimento e cobertura territorial;
3. Os produtos Consumo declarados no código existem, têm chaves únicas e atendem aos testes de cobertura territorial;
4. Os manifestos e metadados obrigatórios estão presentes;
5. Todos os testes integrados apresentam `ok = True`.

### Observações para interpretação

- **Faixa etária em 2024 e 2025:** a nota original informa que o produto usa somente `participantes–PROVA`, pois `TP_FAIXA_ETARIA` está ausente da base Silver `resultados` nesses anos. A regra de cobertura específica do produto de sexo/faixa etária, porém, é implementada no código por listas vazias para 2024 e 2025.
- **Quantidade de produtos:** a documentação anterior menciona oito produtos Analytic e três produtos Consumo, enquanto os dicionários de chaves usados pelos testes contêm, respectivamente, nove e quatro nomes. A descrição foi alinhada ao conteúdo executável, sem alterar o código.
- **Quantidade de fontes Gold:** o teste exige 48 registros no manifesto. A lista de referências territoriais esperadas é um teste separado e não determina, por si só, essa quantidade total.

Antes de encerrar o projeto, recomenda-se conferir essas divergências entre descrições, nomes dos testes e conjuntos efetivamente declarados, caso se pretenda atualizar também a lógica do notebook. Esta versão apenas documenta o comportamento existente.
